# Bernstein-Vazirani on AWS Braket (Rigetti Ankaa-3)

Runs BV for n = 2-8 qubits on Rigetti hardware and adds the results to
`1_combined_dsr_comparison_aws.png` (currently only Grover/QFT). Run the
cells top to bottom.

In [ ]:
# One-time setup: installs qward (with AWS Braket / Rigetti support) from PyPI.
%pip install -q "qiskit-qward[aws]"

In [ ]:
import os
from pathlib import Path

from qward.examples.papers.bv.bv_aws import BVAWSExperiment

## 1. AWS credentials

Fill in your Braket-enabled AWS keys below. No shell env vars or `.env`
file needed — this cell sets everything the executor needs for the rest
of the notebook.

In [ ]:
# Fill these in with your own AWS Braket-enabled credentials.
AWS_ACCESS_KEY_ID = "YOUR_AWS_ACCESS_KEY_ID"
AWS_SECRET_ACCESS_KEY = "YOUR_AWS_SECRET_ACCESS_KEY"
AWS_REGION = "us-west-1"
AWS_DEVICE = "Ankaa-3"

# Set as process env vars so boto3 / the Braket SDK pick them up automatically.
os.environ["AWS_ACCESS_KEY_ID"] = AWS_ACCESS_KEY_ID
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_ACCESS_KEY
os.environ["AWS_DEFAULT_REGION"] = AWS_REGION

## 2. Sanity check (BV2-ALT)

Cheapest config first, to confirm credentials and device access work
end to end before spending on the full campaign.

In [ ]:
# shots/timeout apply to every config run in this notebook.
experiment = BVAWSExperiment(shots=1024, timeout=600)

sanity_result = experiment.run(
    config_id="BV2-ALT",
    device_id=AWS_DEVICE,
    region=AWS_REGION,
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
)
print("Status:", sanity_result["status"])
print("Mean DSR (Michelson):", sanity_result["batch_summary"].get("mean_dsr_michelson"))

## 3. Full campaign (n = 2-8)

Runs `BV{n}-ALT` for each n. Each result is saved automatically under
`bv/data/qpu/aws/`.

In [ ]:
QUBIT_RANGE = range(2, 9)  # BV2-ALT .. BV8-ALT

campaign_status = {}
for n in QUBIT_RANGE:
    config_id = f"BV{n}-ALT"
    print(f"\n=== {config_id} ===")
    try:
        result = experiment.run(
            config_id=config_id,
            device_id=AWS_DEVICE,
            region=AWS_REGION,
            aws_access_key_id=AWS_ACCESS_KEY_ID,
            aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
        )
        campaign_status[config_id] = result["status"]
    except Exception as exc:
        # Keep going through the remaining qubit counts even if one job fails.
        campaign_status[config_id] = f"error: {exc}"

campaign_status

## 4. Verify saved data

Any job still `pending`/`timeout` here can be resolved later by re-running
this notebook, or via `bv_aws.py --update`.

In [ ]:
experiment.print_data_status()

## 5. Rebuild `DSR_result.csv` and regenerate the AWS plot

Same pipeline used for the paper: enrich the new JSON files, rebuild the
unified CSV, then regenerate the figures (including
`1_combined_dsr_comparison_aws.png`, now with BV).

In [ ]:
import subprocess
import sys

import qward.examples.papers as papers_pkg
from IPython.display import Image, display

PAPERS_DIR = Path(papers_pkg.__file__).resolve().parent

# Order matters: enrich the raw JSON first, then fold it into the CSV, then plot.
# Uses this same Python/qward install (no uv or repo checkout required).
for script, script_args in (
    ("enrich_dsr_profile.py", ["--dataset", "bv-aws"]),
    ("enrich_hellinger.py", ["--dataset", "bv-aws"]),
    ("build_csv_from_json.py", []),
    ("differential_success_rate_analysis.py", []),
):
    subprocess.run([sys.executable, script, *script_args], cwd=PAPERS_DIR, check=True)

display(Image(filename=str(PAPERS_DIR / "plots" / "1_combined_dsr_comparison_aws.png")))